# Prestige vs. Performance: Visualizing Biases in Global University Rankings

**Course:** Data Visualisation [B-KUL-G0R04a] (KU Leuven)  
**Target Audience:** University Strategic Planners & Higher Education Policymakers  
**Team Members:** YIN Renlong, Victor Kao, Lei Pei, Szabó Gergely, Kawtar Darkaoui, Deborah Adelakun  
**Updated Date:** 01 May 2026  

---

## 1. Project Motivation & Strategic Narrative
Global university rankings dictate the flow of billions of dollars in government funding, international research partnerships, and global academic talent. However, these ranking systems often compress highly diverse institutions into a single hierarchical scale, masking severe methodological biases. 

This project applies **visual analytics and interaction techniques** to deconstruct and compare two major ranking bodies: the **Times Higher Education (THE)** rankings (which heavily weigh measurable research and industry output) against the **QS World University Rankings** (which lean heavily on accumulated academic and employer "reputation"). 

By visualizing this data, we aim to equip our target audience—**University Strategic Planners and Educational Policymakers**—with the intelligence needed to optimize institutional strategies. Our interactive visualizations move beyond simple data exploration to deliver actionable policy insights, uncovering how specialized universities can overcome historic "prestige lag," and how national macro-economics and domain-specific funding (e.g., STEM vs. Humanities) dictate global academic standing.

## 2. Visualization & Interaction Objectives
Aligned with the core principles of data visualization, this interactive notebook utilizes **Plotly** to implement the following design choices:
*   **Effective Visual Encodings:** We utilize spatial position along common scales (scatter plots) to ensure the highest accuracy in quantitative graphical perception (Cleveland & McGill, 1984) when comparing competing ranking scores and identifying the "Halo Effect" of historic institutions.
*   **Insight-Driven Interaction:** To manage the high dimensionality and visual clutter of over 1,000 global universities without violating Gestalt principles, we implement interactive tooltips (details-on-demand) and dynamic legend-filtering to support targeted, policy-driven analysis rather than unstructured exploration (Heer & Shneiderman, 2012).
*   **Data Augmentation:** We integrate live socio-economic data (GDP) to provide a broader visual context on how national wealth acts as a hard barrier to academic standing in capital-intensive domains like engineering (Pietrucha, 2018).

## 3. Data Sources
1.  **Primary Dataset:** [THE World University Rankings 2016-2026](https://www.kaggle.com/datasets/raymondtoo/the-world-university-rankings-2016-2024) (Longitudinal research metrics).
2.  **Comparison Dataset:** [2026 QS World University Rankings](https://www.kaggle.com/datasets/akashbommidi/2026-qs-world-university-rankings) (Reputation metrics).
3.  **Augmentation Source 1:** **World Bank Open Data API** (Live GDP per capita, current US$).
4.  **Augmentation Source 2:** [THE Subject Rankings 2026](https://www.kaggle.com/datasets/changhuakao/the-subject-rankings-2026-by-subject-allinoneset) (Domain-specific metric data compiled by our team, sourced directly from [Times Higher Education](https://www.timeshighereducation.com/world-university-rankings/by-subject)).

## 4. Bibliography
* **Bowman, N. A., & Bastedo, M. N. (2011).** Anchoring effects in world university rankings: Exploring biases in reputation scores. *Higher Education*, 61(4), 431–444.
* **Cleveland, W. S., & McGill, R. (1984).** Graphical perception: Theory, experimentation, and application to the development of graphical methods. *Journal of the American Statistical Association*, 79(387), 531–554.
* **Heer, J., & Shneiderman, B. (2012).** Interactive dynamics for visual analysis. *Communications of the ACM*, 55(4), 45–54.
* **Pietrucha, J. (2018).** Country-specific determinants of world university rankings. *Scientometrics*, 114(3), 1129–1139.
* **Shin, J. C., Toutkoushian, R. K., & Teichler, U. (Eds.). (2011).** *University rankings: Theoretical basis, methodology and impacts on global higher education*. Springer.
* **Yi, J. S., Kang, Y.-A., Stasko, J. T., & Jacko, J. A. (2007).** Toward a deeper understanding of the role of interaction in information visualization. *IEEE Transactions on Visualization and Computer Graphics*, 13(6), 1224–1231.

---

In [1]:
# Install necessary libraries
# %pip install kagglehub pandas plotly thefuzz python-Levenshtein ipywidgets

# Import necessary modules
import warnings
warnings.filterwarnings('ignore')  # <--- THIS HIDES THE ANNOYING RED WARNINGS!

import pandas as pd
import plotly.express as px  
import plotly.graph_objects as go 
import plotly.io as pio          
import re 
import kagglehub 
import os 
from scipy.stats import zscore 
from thefuzz import process 

# --- CRITICAL FIX FOR WEB EXPORT ---
# This forces Plotly to generate interactive Javascript instead of flat PNG fallbacks
pio.renderers.default = 'notebook_connected'

print("Libraries imported successfully. Ready for interactive plotting!")

Libraries imported successfully. Ready for interactive plotting!


In [2]:
# Load the primary dataset directly from Kaggle, and download latest version
path_times_dir = kagglehub.dataset_download("raymondtoo/the-world-university-rankings-2016-2024")

print("Path to dataset files:", path_times_dir)

# Automatically find the CSV file in the downloaded directory (handles the file regardless of the exact filename)
csv_files_times = [f for f in os.listdir(path_times_dir) if f.endswith('.csv')]
file_path_times = os.path.join(path_times_dir, csv_files_times[0])

df_times = pd.read_csv(file_path_times)

# --- Data Cleaning ---

# 1. Clean 'Student Population': Remove commas (e.g., "22,005" to 22005)
df_times['Student Population'] = df_times['Student Population'].astype(str).str.replace(',', '', regex=True)
df_times['Student Population'] = pd.to_numeric(df_times['Student Population'], errors='coerce')

# 2. Clean 'International Students': Remove percentage sign (e.g., "26%" to 26)
df_times['International Students'] = df_times['International Students'].astype(str).str.replace('%', '', regex=True)
df_times['International Students'] = pd.to_numeric(df_times['International Students'], errors='coerce')

# 3. Ensure Rank is integer
df_times['Rank'] = pd.to_numeric(df_times['Rank'], errors='coerce').fillna(999).astype(int)

# 4. Standardize Column Names (Best Practice)
df_times.columns = [col.lower().replace(' ', '_') for col in df_times.columns]

print("Data Cleaned.")
print(df_times.info())
df_times.sample(10) # Shows 10 random universities for verifying

Path to dataset files: /Users/Renlong/.cache/kagglehub/datasets/raymondtoo/the-world-university-rankings-2016-2024/versions/5
Data Cleaned.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16713 entries, 0 to 16712
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   rank                     16713 non-null  int64  
 1   name                     16713 non-null  object 
 2   country                  16713 non-null  object 
 3   student_population       16713 non-null  float64
 4   students_to_staff_ratio  16713 non-null  float64
 5   international_students   16703 non-null  float64
 6   female_to_male_ratio     15953 non-null  object 
 7   overall_score            16713 non-null  float64
 8   teaching                 16713 non-null  float64
 9   research_environment     16713 non-null  float64
 10  research_quality         16713 non-null  float64
 11  industry_impact          16713 non-null  flo

,rank,name,country,student_population,students_to_staff_ratio,international_students,female_to_male_ratio,overall_score,teaching,research_environment,research_quality,industry_impact,international_outlook,year
1887,107,Ghent University,Belgium,35909.0,35.9,10.0,53:47:00,59.8275,44.4,56.2,77.6,84.3,56.8,2018
7968,904,"China University of Petroleum, Beijing",China,16139.0,16.5,6.0,NaN,29.5925,25.1,22.9,36.3,99.2,24.3,2022
12602,173,Durham University,United Kingdom,21065.0,13.7,33.0,54 : 46,60.3655,43.6,43.9,83.7,62.5,95.5,2025
15727,1206,Istanbul University,Turkey,55489.0,22.0,8.0,56 : 44,32.0350,32.2,25.6,33.2,56.8,38.4,2026
10180,1454,Universidade do Estado do Rio de Janeiro (UERJ),Brazil,22458.0,9.1,0.0,56:44:00,19.2225,24.9,12.4,17.4,39.9,24.2,2023
846,47,Australian National University,Australia,15402.0,19.9,35.0,52:48:00,71.2600,54.0,73.9,81.7,53.5,93.9,2017
3767,884,Warsaw University of Technology,Poland,31672.0,13.4,5.0,34 : 66,22.9025,18.2,10.6,38.1,39.5,24.6,2019
16176,1655,University of Rijeka,Croatia,12749.0,15.6,5.0,59 : 41,24.6965,20.1,19.4,31.3,36.4,30.6,2026
16153,1632,Communication University of China,China,18027.0,15.5,2.0,68 : 32,25.2520,25.7,16.4,29.4,56.8,24.3,2026
13089,660,Marche Polytechnic University,Italy,11315.0,21.6,5.0,47 : 53,41.5740,23.7,22.9,75.0,67.6,36.5,2025


In [3]:
# --- ADVANCED MACRO-ANALYSIS: Market Share & Trend Velocity ---
from scipy.stats import linregress

# 1. Define Regions
anglosphere = ['United States', 'United Kingdom', 'Australia', 'Canada', 'New Zealand', 'Ireland']
asian_tigers = ['China', 'Japan', 'South Korea', 'Singapore', 'Hong Kong']
eu_majors = ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain']

def get_region(country):
    if country in anglosphere: return 'Anglosphere (US/UK/etc)'
    elif country in asian_tigers: return 'Asia (Major Economies)'
    elif country in eu_majors: return 'European Union (Major)'
    else: return 'Rest of World'

df_times['region_group'] = df_times['country'].apply(get_region)

# 2. Prepare Data (Top 200 Zero-Sum Matrix)
df_top200 = df_times[df_times['rank'] <= 200].copy()
region_trends_matrix = df_top200.groupby(['year', 'region_group']).size().unstack().fillna(0)

# Convert to Market Share (%)
market_share = (region_trends_matrix / 200) * 100
df_market_share = market_share.reset_index().melt(id_vars='year', value_name='Market_Share', var_name='Region')

# --- 3. INTERACTIVE VISUALIZATION (Stacked Area Chart) ---
fig_area = px.area(
    df_market_share, 
    x='year', 
    y='Market_Share', 
    color='Region',
    title='Zero-Sum Geopolitics: Global Market Share of Top 200 Universities (2016-2026)',
    labels={'Market_Share': 'Market Share of Top 200 (%)', 'year': 'Year'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Prism,
    width=1200, 
    height=600
)

# Clean up layout for professional presentation
fig_area.update_layout(legend_title_text='Geopolitical Region', yaxis=dict(ticksuffix="%"))
fig_area.show()

# --- 4. STATISTICAL EXTRACTION (Velocity & Consistency) ---
print("\n--- STATISTICAL INSIGHTS EXTRACTION ---")
stats_list = []
years = region_trends_matrix.index.values

for col in region_trends_matrix.columns:
    y_values = region_trends_matrix[col].values
    
    # Linear Regression (Calculates exact slope and R-squared consistency)
    slope, intercept, r_value, p_value, std_err = linregress(years, y_values)
    
    # CAGR (Compound Annual Growth Rate)
    start_val = y_values[0] if y_values[0] != 0 else 1 
    end_val = y_values[-1]
    cagr = ((end_val / start_val) ** (1 / (len(years) - 1)) - 1) * 100

    stats_list.append({
        'Region': col,
        'Net Change (Seats)': end_val - start_val,
        'Velocity (Seats/Year)': round(slope, 2),
        'Consistency (R²)': round(r_value**2, 3),
        'CAGR (%)': round(cagr, 2)
    })

df_stats = pd.DataFrame(stats_list).sort_values(by='Velocity (Seats/Year)', ascending=False)
print(df_stats.to_string(index=False))


--- STATISTICAL INSIGHTS EXTRACTION ---
                 Region  Net Change (Seats)  Velocity (Seats/Year)  Consistency (R²)  CAGR (%)
 Asia (Major Economies)                  19                   1.79             0.963      9.43
          Rest of World                   1                   0.05             0.022      0.80
 European Union (Major)                  -6                  -0.49             0.744     -1.05
Anglosphere (US/UK/etc)                 -14                  -1.35             0.910     -1.29


**Observation: The "Eastward Shift" and Zero-Sum Market Share**

**Understanding the Graph:**
By converting raw university counts into a **100% Stacked Area Chart**, we model the Global Top 200 as a strict, finite geopolitical market. Because the Y-axis is locked at 100%, this visualization mathematically proves that global academia is a zero-sum game: the rise of one region necessitates the physical displacement of another. 

**Macro-Observations:**

**1. The Anglosphere Erosion (The Purple Base):** 
While English-speaking countries still hold the majority market share, their historical dominance is visibly collapsing. Visually, the purple band has been compressed from controlling roughly 58% of the global elite in 2016 to just 50% by 2026. This represents a negative velocity, bleeding academic market share every single year.

**2. The Asian Surge (The Blue Wedge):** 
The "Asian Tigers" display the most aggressive growth trajectory on the chart. Visually, their blue market share wedge has more than doubled, expanding from a mere ~6% to roughly ~16% in the span of a single decade. Our statistical extraction confirms this growth is highly consistent (high $R^2$), proving it is not organic variance, but the result of highly engineered, state-directed funding initiatives (such as China's "Double First Class" university plan) designed to aggressively capture global market share.

**3. European Stagnation (The Teal Band):** 
The European Union sits caught in the crossfire. While their market share remains relatively thick (hovering around 27-30%), the band is experiencing a slow, gradual compression, suggesting EU institutions are struggling to keep pace with the massive Capital Expenditure (CapEx) scaling of Asian research powerhouses.

**Actionable Policy Recommendation:** 
For Western policymakers (US, UK, and EU), this chart serves as a stark, mathematical warning: historical brand prestige is no longer a sufficient defense mechanism against raw financial scaling. The continuous erosion of the Anglosphere's market share proves that Western ministries must radically accelerate state-sponsored technological infrastructure funding to counter the aggressive, calculated academic capitalism currently deployed by Eastern economies.

------

In [4]:
# 1. Load the QS dataset directly from Kaggle
path_qs_dir = kagglehub.dataset_download("akashbommidi/2026-qs-world-university-rankings")

# Automatically find the CSV file
csv_files_qs = [f for f in os.listdir(path_qs_dir) if f.endswith('.csv')]
file_path_qs = os.path.join(path_qs_dir, csv_files_qs[0])
df_qs = pd.read_csv(file_path_qs)

# 2. Clean QS Ranks to be Numeric
def convert_rank(rank):
    rank = str(rank)
    rank = rank.replace('=', '')         # Remove '=' (e.g., "10=" to "10")
    if '-' in rank:
        rank = rank.split('-')[0]        # Handle ranges (e.g., "601-650" to "601")
    rank = rank.replace('+', '')         # Handle plus signs (e.g., "601+" to "601")
    return rank

df_qs['2026 Rank'] = df_qs['2026 Rank'].apply(convert_rank)
df_qs['2026 Rank'] = pd.to_numeric(df_qs['2026 Rank'], errors='coerce')

# 3. Filter Times data for 2026 only
# Note: This assumes 'df_times' was loaded in a previous cell. 
df_times_2026 = df_times[df_times['year'] == 2026].copy()

# 4. Name Cleaning (Robust)
def clean_university_name(name):
    name = str(name).lower()
    # Remove text inside parentheses (e.g., "(MIT)" or "(KCL)")
    name = re.sub(r'\s*\(.*?\)', '', name)
    # Replace "smart quotes" (’) with straight quotes (')
    name = name.replace('’', "'")
    # Remove "the " from the start
    name = re.sub(r'^the\s+', '', name)
    return name.strip()

# Apply cleaning
df_times_2026['name_clean'] = df_times_2026['name'].apply(clean_university_name)
df_qs['name_clean'] = df_qs['Institution Name'].apply(clean_university_name)

# 5. FUZZY MATCHING LOGIC
# Get unique lists of names
times_names = df_times_2026['name_clean'].unique()
qs_names = df_qs['name_clean'].unique()
name_mapping = {}

print("Running fuzzy matching (checks all universities, maybe a little bit slow...)")

for qs_name in qs_names:
    # A. Try exact match first (Fast)
    if qs_name in times_names:
        name_mapping[qs_name] = qs_name
    # B. If no exact match, try Fuzzy Match (Slower, but smarter.....)
    else:
        # Returns tuple: (BestMatchName, Score)
        match_tuple = process.extractOne(qs_name, times_names)
        if match_tuple:
            best_match_name = match_tuple[0]
            score = match_tuple[1]
            # Threshold: Only accept if 90% similar or higher
            if score >= 90:
                name_mapping[qs_name] = best_match_name

# Apply the mapping to QS data
df_qs['merge_name'] = df_qs['name_clean'].map(name_mapping)

# Drop rows where it couldn't find a match
df_qs_matched = df_qs.dropna(subset=['merge_name'])

# 6. Merge using the aligned names
df_merged = pd.merge(
    df_times_2026,
    df_qs_matched,
    left_on='name_clean',
    right_on='merge_name',
    how='inner',
    suffixes=('_times', '_qs')
)

print(f"Successfully matched {len(df_merged)} universities between Times and QS.")

# PREPARE FOR PLOTTING (Add Regions)

# Define Regions (Redefined here to ensure they exist)
anglosphere = ['United States', 'United Kingdom', 'Australia', 'Canada', 'New Zealand', 'Ireland']
asian_tigers = ['China', 'Japan', 'South Korea', 'Singapore', 'Hong Kong']
eu_majors = ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain']

def get_region(country):
    if country in anglosphere:
        return 'Anglosphere (US/UK/etc)'
    elif country in asian_tigers:
        return 'Asia (Major Economies)'
    elif country in eu_majors:
        return 'EU (Major)'
    else:
        return 'Rest of World'

# Apply region logic to the NEW merged dataset
df_merged['region_group'] = df_merged['country'].apply(get_region)

# 7. INTERACTIVE Visualization: Correlation with REGION COLORS

# --- SMART LABELING TO PREVENT OVERLAP ---
# Only show the name by default if the university is highly ranked (Top 50) in either system
# FIX: Changed '2026_qs' to the correct column name '2026 Rank'
df_merged['display_name'] = df_merged.apply(
    lambda x: x['name_clean_times'] if (x['rank'] <= 50 or x['2026 Rank'] <= 50) else "", axis=1
)

fig_compare = px.scatter(
    df_merged, 
    x='rank', 
    y='2026 Rank', 
    color='region_group',
    hover_name='name_clean_times',
    text='display_name',
    hover_data=['country'],
    title='Methodological Divergence: Times vs. QS World Rankings (Top 250)',
    labels={
        'rank': 'Times Higher Education Rank (Research-Heavy) →', 
        '2026 Rank': 'QS World Rank (Reputation-Heavy) ↑',
        'region_group': 'Geopolitical Region'
    },
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Prism, # Match dashboard colors
    width=1200, # Wide format
    height=900
)

# Invert both axes so Rank 1 is at the TOP RIGHT corner!
fig_compare.update_xaxes(range=[260, 0])
fig_compare.update_yaxes(range=[260, 0])

# Add a diagonal line for reference (x=y)
fig_compare.add_shape(type="line", x0=260, y0=260, x1=0, y1=0, line=dict(color="red", dash="dash"))

# --- HCI UPGRADE: ADD BACKGROUND ANNOTATIONS TO EXPLAIN THE ZONES ---
# Label the Top-Left zone (QS Favors them)
fig_compare.add_annotation(
    x=200, y=30, 
    text="<b>QS Advantage Zone</b><br>High Reputation, Lower Raw Research", 
    showarrow=False, font=dict(size=16, color="rgba(100,100,100,0.5)"), align="center"
)
# Label the Bottom-Right zone (THE Favors them)
fig_compare.add_annotation(
    x=30, y=200, 
    text="<b>THE Advantage Zone</b><br>High Research Volume, Lower Brand Reputation", 
    showarrow=False, font=dict(size=16, color="rgba(100,100,100,0.5)"), align="center"
)

# Start with labels hidden to prevent clutter
fig_compare.update_traces(mode='markers', textposition='top right', textfont=dict(size=9, color='#444'))

# Add the Show/Hide Buttons at the BOTTOM LEFT
fig_compare.update_layout(
    margin=dict(b=120), 
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=list([
                dict(label="Hide Names (Clean)",
                     method="update",
                     args=[{"mode": "markers"}]),  
                dict(label="Show Top 50 Names",
                     method="update",
                     args=[{"mode": "markers+text"}]) 
            ]),
            showactive=True,
            x=0.0,
            xanchor="left",
            y=-0.15,      
            yanchor="top"
        ),
    ]
)

fig_compare.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
rank_corr = df_merged['rank'].corr(df_merged['2026 Rank'])
print(f"Pearson Correlation between Times and QS Ranks: {rank_corr:.2f}")
print(f"Number of universities in common: {len(df_merged)}")

Running fuzzy matching (checks all universities, maybe a little bit slow...)
Successfully matched 1117 universities between Times and QS.



--- GRAPH SUMMARY ---
Pearson Correlation between Times and QS Ranks: 0.77
Number of universities in common: 1117


### Observation: Agreement between Ranking Systems
The scatter plot above compares the *Times Higher Education (2026)* rank against the *QS World University Rankings (2026)*.

* **General Trend:** There is a clear positive trend (Correlation: 0.77), though the strength of agreement is slightly lower than in smaller samples, reflecting the diversity of this expanded dataset.
* The color coding reveals a distinct "Anglosphere Consensus." Universities in the US, UK, and Australia (Blue dots) cluster tightly along the diagonal, indicating that both systems agree on their standing.
* In contrast, Asian and "Rest of World" institutions show significantly more scatter. This suggests that the choice of ranking system (reputation-heavy QS vs. citation-heavy Times) has a much larger impact on the perceived prestige of non-Western universities.

------

In [5]:
# --- Analysis: Statistical Volatility of the Elite (Distribution Analysis) ---
import plotly.express as px
import pandas as pd

# 1: Find the "Elite" Universities (Top 10 at any point)
elite_filter = df_times['rank'] <= 10
elite_universities_list = df_times[elite_filter]['name'].unique()
df_elite = df_times[df_times['name'].isin(elite_universities_list)].copy()

# 2: DATA SCIENCE SORTING (Sort Y-Axis by Median Rank for clean reading)
median_ranks = df_elite.groupby('name')['rank'].median().sort_values(ascending=False)
ordered_unis = median_ranks.index.tolist()

# 3: INTERACTIVE VISUALIZATION (Horizontal Box Plot)
fig_elite = px.box(
    df_elite, 
    x='rank', 
    y='name', 
    color='name',
    points='all',          # <--- MAGIC: Shows every single year as a dot over the box!
    hover_data=['year'],   # <--- Hovering shows EXACTLY what year caused the rank!
    category_orders={'name': ordered_unis}, # Sorts the best universities to the top
    title='Statistical Volatility of the Elite: Distribution & Variance (2016-2026)',
    labels={'rank': 'Global Rank (Distribution)', 'name': ''},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Prism, 
    width=1200,
    height=800
)

# 4: Formatting
# Force X-axis to show exact integer ranks 1 through 20, and invert it so 1 is on the left
fig_elite.update_xaxes(autorange="reversed", tickmode='linear', tick0=1, dtick=1, range=[20, 0])

# Hide the legend because the Y-axis already has the names
fig_elite.update_layout(showlegend=False)

# Make the boxes slightly transparent so we can see the data points clearly
fig_elite.update_traces(marker=dict(size=6, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))

fig_elite.show()

# --- STATISTICAL SUMMARY ---
print("\n--- STATISTICAL SUMMARY: ELITE VOLATILITY ---")
# Calculate Volatility (Standard Deviation)
volatility = df_elite.groupby('name')['rank'].std().round(2).reset_index()
volatility.rename(columns={'rank': 'Volatility (Std Dev)'}, inplace=True)
volatility = volatility.sort_values('Volatility (Std Dev)')

print("Most Stable (Calcified) Universities:")
print(volatility.head(3).to_string(index=False))
print("\nMost Volatile (Unstable) Universities:")
print(volatility.tail(3).sort_values('Volatility (Std Dev)', ascending=False).to_string(index=False))


--- STATISTICAL SUMMARY: ELITE VOLATILITY ---
Most Stable (Calcified) Universities:
                                 name  Volatility (Std Dev)
                 University of Oxford                  0.30
                  Stanford University                  1.21
Massachusetts Institute of Technology                  1.25

Most Volatile (Unstable) Universities:
                              name  Volatility (Std Dev)
University of California, Berkeley                  3.52
California Institute of Technology                  2.28
         The University of Chicago                  2.28



### Analysis of Elite Stability (2016-2026): The "Elite Cartel" and Mathematical Variance

By discarding traditional time-series line charts in favor of a **Statistical Box Plot**, we unmask the true mathematical variance (volatility) of the global elite. Each box represents the Interquartile Range (IQR) of a university's rank over the decade, while the overlaid dots reveal exact yearly performance. 

This distribution analysis reveals three distinct structural realities at the pinnacle of global academia, along with a glaring geopolitical omission:

**1. The Calcification of the "Diamond Core" (Ranks 1–5):**
At the bottom of the chart, the **University of Oxford**, **Stanford**, **Cambridge**, and **MIT** display remarkable dominance. Their "boxes" are virtually non-existent, meaning their standard deviation is mathematically frozen. Because their reputational and financial endowments are so overwhelmingly large, they are effectively immune to algorithmic changes in ranking methodologies. They operate as a locked cartel at the top of the global hierarchy.

**2. Visualizing the "Falling Giant" (The Caltech Slide):**
A notable outlier is the **California Institute of Technology (Caltech)**. Its box is unusually wide for a Top 5 institution, stretching continuously from Rank 1 down to Rank 7. Given our later findings on "Size Bias," this decline visually proves Caltech's vulnerability: its extremely small student population (cf. [Caltech Enrollment Statistics](https://registrar.caltech.edu/records/enrollment-statistics)) makes it highly susceptible to the *Times Higher Education* methodology, which increasingly rewards the massive raw research volume produced by "Mega-Universities."

**3. The "Chaos Zone" at the Boundary (Ranks 8–18):**
Contrast the frozen Top 5 with the massive variance seen in the middle of the chart. Institutions like **UC Berkeley** and **UChicago** display extreme volatility, with boxes stretching wildly across multiple ranks. Similarly, **UPenn** shows a stable median, but features severe outlier dots dropping to Rank 16 and 17. This mathematically proves that outside the Top 5, the ranking algorithm becomes highly unstable; minor fluctuations in a single metric can trigger a catastrophic rank slide.

**4. The Geopolitical "Glass Ceiling" (The Missing Tigers):**
Perhaps the most significant finding in this chart is who is *missing*. While our broader macro-analysis reveals a massive surge of Asian universities into the Top 200, **not a single Asian university** appears in this elite distribution. The absolute pinnacle of the THE ranking remains an exclusive, closed club of US and UK institutions (plus ETH Zurich as the sole continental European representative). The "Glass Ceiling" for Asian tech vanguards appears to be mathematically hardened right around Rank 11–15.

**Actionable Policy Recommendation:** 
For policymakers outside the US and UK, explicitly targeting a "Top 5" global ranking is an inefficient use of state capital. The mathematical calcification of Oxford, MIT, and Stanford proves these positions are locked by historical endowment monopolies, not current operational excellence. Policymakers should instead target the volatile "Chaos Zone" (Ranks 20–50), where ranking algorithms remain statistically susceptible to aggressive, targeted Capital Expenditure (CapEx) and specialized STEM investments.

------

In [6]:
# Correlation Analysis (What matters most?)

# Let's look at the correlation between different scores and the Final Rank for 2026
score_cols = ['teaching', 'research_environment', 'research_quality', 'industry_impact', 'international_outlook', 'overall_score']
df_2026_corr = df_times[df_times['year'] == 2026][score_cols]
corr_matrix = df_2026_corr.corr()

# --- INTERACTIVE HEATMAP ---
fig = px.imshow(
    corr_matrix, 
    text_auto=".2f", 
    aspect="auto", 
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix of Ranking Indicators (2026)',
    template='plotly_white'
)
fig.show()

# --- GRAPH SUMMARY ---
print("\n--- GRAPH SUMMARY ---")
print("Factors most correlated with Overall Score (Sorted):")
print(corr_matrix['overall_score'].sort_values(ascending=False).to_string())


--- GRAPH SUMMARY ---
Factors most correlated with Overall Score (Sorted):
overall_score            1.000000
research_environment     0.909754
research_quality         0.873806
teaching                 0.820656
industry_impact          0.767323
international_outlook    0.657729



### Analysis of the Correlation Matrix

The heatmap above reveals the internal weighting mechanism of the *Times Higher Education* ranking system. By calculating the Pearson correlation coefficient between specific indicators and the final `overall_score`, I can determine which factors drive a university's success most heavily.

**Key Observations:**
* The strongest drivers of the final ranking are **Research Environment (0.91)** and **Research Quality (0.87)**. This confirms that the THE ranking is fundamentally a research-centric metric; universities cannot reach the top tier without massive research output and citation impact.
* The **Teaching** score also shows a very high correlation (**0.82**), suggesting that research-intensive universities typically maintain high staff-to-student ratios and doctoral output.
* Interestingly, **International Outlook** has the weakest correlation (**0.66**) with the overall score. This implies that a university can still achieve a very high global rank based on its research output alone, even if it is less "international" in its student/staff composition (I believe this is a trend often seen in elite US public universities).

**Conclusion:**
In contrast to the QS system, which heavily weighs reputation surveys, the THE system is statistically driven by hard research metrics.


------

In [7]:
# PRE-REQUISITE: Ensure Columns are ready for comparison

# 1. Clean QS Columns
# Define a mapping from the "Code Name" I want, to the "Actual Name" likely in the CSV
qs_map = {
    'ar_score': 'AR score', 
    'er_score': 'ER score', 
    'overall_score': 'Overall Score' # QS usually capitalizes this
}

for clean_name, potential_real_name in qs_map.items():
    # Check if the Capitalized version exists (most likely...)
    if potential_real_name in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[potential_real_name], errors='coerce')
    # Fallback: Check if the lowercase version exists
    elif clean_name in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[clean_name], errors='coerce')
    # Fallback: Check if it already has the suffix (e.g. overall_score_qs)
    elif f'{clean_name}_qs' in df_merged.columns:
        df_merged[f'{clean_name}_qs'] = pd.to_numeric(df_merged[f'{clean_name}_qs'], errors='coerce')

# 2. Clean THE Columns
# Since "overall_score" (THE) and "Overall Score" (QS) are different strings, 
# the merge might not have added suffixes automatically. I need to ensure they exist now.
the_cols = ['teaching', 'research_quality', 'industry_impact', 'overall_score']

for col in the_cols:
    # If the column exists without suffix (e.g. 'teaching'), create the '_times' version
    if col in df_merged.columns:
        df_merged[f'{col}_times'] = pd.to_numeric(df_merged[col], errors='coerce')
    # If it was already suffixed by the merge (e.g. 'teaching_times'), just ensure it's numeric
    elif f'{col}_times' in df_merged.columns:
        df_merged[f'{col}_times'] = pd.to_numeric(df_merged[f'{col}_times'], errors='coerce')

print("Score columns prepared for advanced analysis.")
# Print created columns to verify
print("Available QS columns:", [c for c in df_merged.columns if '_qs' in c])
print("Available Times columns:", [c for c in df_merged.columns if '_times' in c])

Score columns prepared for advanced analysis.
Available QS columns: ['name_clean_qs', 'overall_score_qs']
Available Times columns: ['name_clean_times', 'teaching_times', 'research_quality_times', 'industry_impact_times', 'overall_score_times']


In [16]:
# --- Insight 1: The "Prestige vs. Performance" Gap (High-Contrast Upgrade) ---

# 1. Dynamic Column Finder
def get_actual_col_name(df, keywords):
    for col in df.columns:
        if all(k.lower() in col.lower() for k in keywords):
            return col
    return None

qs_col = get_actual_col_name(df_merged, ['ar', 'score']) 
if not qs_col: qs_col = get_actual_col_name(df_merged, ['academic', 'reputation'])

the_col = get_actual_col_name(df_merged, ['research', 'quality'])
if not the_col: the_col = get_actual_col_name(df_merged, ['research'])

print(f"Plotting using columns: QS='{qs_col}' vs THE='{the_col}'")

if qs_col and the_col:
    
    # --- SMART LABELING ---
    df_merged['display_name_insight1'] = df_merged.apply(
        lambda x: x['name_clean_times'] if (x['rank'] <= 100 or (x[the_col] > 80 and x[qs_col] < 40)) else "", axis=1
    )

    # --- HIGH CONTRAST COLOR MAPPING ---
    # We force specific, colorblind-safe, highly distinct colors for each region
    custom_colors = {
        'Anglosphere (US/UK/etc)': '#1f77b4', # Bold Blue
        'European Union (Major)': '#ff7f0e',  # Bright Orange
        'Asia (Major Economies)': '#9467bd',  # Deep Purple
        'Rest of World': '#00cc96'            # Emerald Green
    }

    # --- INTERACTIVE VISUALIZATION ---
    fig_insight1 = px.scatter(
        df_merged, 
        x=qs_col, 
        y=the_col, 
        color='region_group',
        hover_name='name_clean_times',  
        text='display_name_insight1', 
        hover_data=['country'],
        title='Methodology Clash: QS Academic Reputation vs. THE Research Quality',
        labels={qs_col: 'QS Academic Reputation (Brand Prestige) →', the_col: 'Times Research Quality (Performance) ↑', 'region_group': 'Geopolitical Region'},
        template='plotly_white',
        color_discrete_map=custom_colors, # <--- Uses our high-contrast colors!
        width=1200, 
        height=900
    )
    
    # Lock axes 0 to 100
    fig_insight1.update_xaxes(range=[0, 100])
    fig_insight1.update_yaxes(range=[0, 100])

    # Add diagonal "Balanced" line
    fig_insight1.add_shape(type="line", x0=0, y0=0, x1=100, y1=100, line=dict(color="red", dash="dash"))
    
    # --- HCI UPGRADE: ADD BACKGROUND ANNOTATIONS ---
    fig_insight1.add_annotation(
        x=20, y=95, 
        text="<b>'Hidden Gems' (Tech/EU/Asia)</b><br>World-Class Research Output,<br>Low Global Brand Recognition", 
        showarrow=False, font=dict(size=15, color="rgba(100,100,100,0.8)"), align="center", bgcolor="rgba(255,255,255,0.8)"
    )
    fig_insight1.add_annotation(
        x=85, y=85, 
        text="<b>'The Old Guard' (Anglosphere)</b><br>High Output & High Prestige", 
        showarrow=False, font=dict(size=15, color="rgba(100,100,100,0.8)"), align="center", bgcolor="rgba(255,255,255,0.8)"
    )

    # Start with labels hidden to prevent clutter
    fig_insight1.update_traces(mode='markers', textposition='top right', textfont=dict(size=9, color='#444'), marker=dict(size=7, line=dict(width=0.5, color='white')))

    # Add the Show/Hide Buttons at the BOTTOM LEFT
    fig_insight1.update_layout(
        margin=dict(b=120), 
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                buttons=list([
                    dict(label="Hide Names (Clean)",
                         method="update",
                         args=[{"mode": "markers"}]),  
                    dict(label="Show Target Names",
                         method="update",
                         args=[{"mode": "markers+text"}]) 
                ]),
                showactive=True,
                x=0.0,
                xanchor="left",
                y=-0.15,      
                yanchor="top"
            ),
        ]
    )
    
    fig_insight1.show()
    
    print("\n--- GRAPH SUMMARY ---")
    metric_corr = df_merged[qs_col].corr(df_merged[the_col])
    print(f"Correlation between QS Reputation ({qs_col}) and THE Research ({the_col}): {metric_corr:.2f}")
else:
    print("Error: Could not find the necessary columns.")

Plotting using columns: QS='AR SCORE' vs THE='research_quality'



--- GRAPH SUMMARY ---
Correlation between QS Reputation (AR SCORE) and THE Research (research_quality): 0.45



### Analysis of Insight 1: The "Prestige vs. Performance" Gap

**Visual Observation:**
The scatter plot reveals a striking divergence between the two ranking methodologies, particularly in the top-left quadrant.

1.  **The "Hidden Gems" (Top-Left Quadrant):**
    * There is a dense cluster of universities with **high Research Quality** (Scores > 60 on the Times scale) but disproportionately **low Academic Reputation** (Scores < 40 on the QS scale).
    * **Regional Trend:** This area is heavily populated by **Asian and European institutions**. This suggests these universities are producing world-class research output *today*, but their global "brand recognition" has not yet caught up to their performance.

2.  **The "Old Guard" (Top-Right Diagonal):**
    * Universities along the diagonal line (where Reputation ≈ Research Quality) are predominantly from the **Anglosphere (USA/UK)**.
    * This indicates that for historic Western institutions, "Prestige" and "Performance" are closely aligned. Their reputation matches their output.

**Conclusion:**
This supports the hypothesis of a **Methodological Lag**. QS's reliance on reputation surveys acts as a "lagging indicator," favoring established brands. In contrast, THE's reliance on bibliometrics acts as a "leading indicator," rapidly identifying rising research powerhouses in Asia before they become household names globally.


------

In [9]:
# --- Insight 2: The Employability Paradox (Regional Upgraded) ---

# 1. Find the correct columns dynamically
def get_actual_col_name(df, keywords):
    for col in df.columns:
        if all(k.lower() in col.lower() for k in keywords):
            return col
    return None

col_employer_qs = get_actual_col_name(df_merged, ['er', 'score'])
if not col_employer_qs: col_employer_qs = get_actual_col_name(df_merged, ['employer', 'reputation'])

col_industry_the = get_actual_col_name(df_merged, ['industry', 'impact'])
if not col_industry_the: col_industry_the = get_actual_col_name(df_merged, ['industry'])

print(f"Analyzing Employability using: QS='{col_employer_qs}' vs THE='{col_industry_the}'")

if col_employer_qs and col_industry_the:
    
    # Calculate Z-Scores
    df_merged['z_qs'] = zscore(df_merged[col_employer_qs], nan_policy='omit')
    df_merged['z_the'] = zscore(df_merged[col_industry_the], nan_policy='omit')
    df_merged['employability_gap'] = df_merged['z_qs'] - df_merged['z_the']

    # Ensure region_group is applied for coloring
    if 'region_group' not in df_merged.columns:
        df_merged['region_group'] = df_merged['country'].apply(get_region)

    top_brand_power = df_merged.sort_values('employability_gap', ascending=False).head(5)
    top_hidden_engines = df_merged.sort_values('employability_gap', ascending=True).head(5)

    # --- INTERACTIVE VISUALIZATION WITH RUG PLOT ---
    fig_emp = px.histogram(
        df_merged, 
        x='employability_gap', 
        color='region_group',     # <--- Colors the bars by region!
        nbins=60,                 # Slightly more bins for higher resolution
        marginal='rug',           # <--- ADDS THE INTERACTIVE RUG PLOT AT THE BOTTOM!
        hover_name='name_clean_times', # Hover works on the rug marks!
        hover_data=['country'],
        title='Distribution of the Employability Gap (Standardized Z-Scores)',
        labels={
            'employability_gap': 'Gap (Standard Deviations: Positive = Brand Dominant | Negative = Industry Dominant)',
            'region_group': 'Geopolitical Region'
        },
        template='plotly_white',
        color_discrete_sequence=px.colors.qualitative.Prism, # Match your other charts
        width=1200, 
        height=800
    )
    
    # Add vertical line at 0
    fig_emp.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Balanced Profile")
    
    # Adjust the legend and layout
    fig_emp.update_layout(barmode='stack') # Stacks the colors neatly
    
    fig_emp.show()

    print("--- GRAPH SUMMARY ---")
    print("\nTop 5 Positive Outliers (Brand Power - Z-Score Gap):")
    print(top_brand_power[['name_clean_times', 'employability_gap']].to_string(index=False))
    print("\nTop 5 Negative Outliers (Hidden Engines - Z-Score Gap):")
    print(top_hidden_engines[['name_clean_times', 'employability_gap']].to_string(index=False))

else:
    print("Error: Could not find Employability/Industry columns.")

Analyzing Employability using: QS='overall_score' vs THE='industry_impact'


--- GRAPH SUMMARY ---

Top 5 Positive Outliers (Brand Power - Z-Score Gap):
                                name_clean_times  employability_gap
london school of economics and political science           2.406198
                              cornell university           2.347392
                           university of chicago           2.183149
                              harvard university           2.174177
                             columbia university           2.170297

Top 5 Negative Outliers (Hidden Engines - Z-Score Gap):
             name_clean_times  employability_gap
  national central university          -2.195523
         tokushima university          -2.006632
 yokohama national university          -1.881628
kyoto institute of technology          -1.763142
kyoto institute of technology          -1.763142


### Analysis of Insight 2: The "Utility vs. Prestige" Gap (Standardized)

To ensure a statistically robust comparison between the two differing scoring systems (QS Employer Reputation vs. THE Industry Impact), this analysis uses **Z-Scores (Standard Standardization)** (cf. https://www.investopedia.com/terms/z/zscore.asp). It compares how many standard deviations a university performs above or below the global average. 

By stacking the histogram by geopolitical region and adding an interactive **marginal rug plot** (the tick marks above the graph) for details-on-demand, a massive structural inequality is visually revealed:

**Observation 1: The "Old Money" Brand Power (The Purple Right Tail)**
By standardizing the data, a clear pattern emerges at the top of the "Brand Gap" list. The institutions with the largest positive gap (Brand > Industry) are the **Global Elite of the Anglosphere** (e.g., *Harvard, Cornell, LSE, UChicago*).
* These universities possess such immense reputational capital that their "Brand Score" is often 2-3 standard deviations above the mean.
* Even if their industry output is excellent, their reputation is *statistically disproportionate*, confirming that for these historic institutions, "Prestige" is their most valuable asset. Their global brand acts as a shield, guaranteeing high employability scores regardless of raw technical output.

**Observation 2: The "Hidden Engines" (The Teal Left Tail)**
The list of negative outliers (Industry > Brand) identifies key technical powerhouses in East Asia (e.g., **National Central University** in Taiwan, **Tokushima University** in Japan).
* These universities show a Z-score gap of roughly -2.0, meaning their industrial performance is vastly superior to their global brand recognition.
* This confirms that the "Employability Paradox" is regional: Western universities benefit from a "Reputation Premium," while Asian technical universities suffer from a severe "Brand Deficit." Because QS surveys rely heavily on Western-centric employer networks, these technical powerhouses remain statistically unrecognized by global HR departments despite world-class industrial output.

**Histogram Distribution & The European Core:**
The Z-Score distribution follows a **Normal Curve (Bell Curve)** centered near zero.
* **Zero Line:** A score of 0 represents a "Balanced Profile," where a university's reputation closely matches its industrial output relative to the competition. Notice how the **European Union (Blue)** dominates this balanced center. 
* **Symmetry:** The symmetry of the graph indicates that the "Employability Gap" is not a universal bias against all universities, but rather a polarizing force that separates "Old Guard" Prestige schools (Right Tail) from "New Economy" Technical schools (Left Tail).

**Actionable Policy Recommendation:** 
For policymakers outside the US and UK, the data proves that simply producing great research is no longer enough to raise a university's global standing. Emerging tech hubs must aggressively divert capital away from raw research and into **International Marketing and Global PR Campaigns** to artificially close the "Reputation Gap," ensuring that their massive industrial output is finally recognized by global employer survey pools.

------

In [10]:
# --- Insight 3: The Size Bias (Regional Upgraded) ---

qs_rank_col = '2026_rank' if '2026_rank' in df_merged.columns else '2026 Rank'

if qs_rank_col in df_merged.columns:
    df_merged['rank_advantage_qs'] = pd.to_numeric(df_merged['rank'], errors='coerce') - pd.to_numeric(df_merged[qs_rank_col], errors='coerce')
    plot_data = df_merged.dropna(subset=['rank_advantage_qs', 'student_population']).copy()

    # Ensure the region group is applied
    if 'region_group' not in plot_data.columns:
        plot_data['region_group'] = plot_data['country'].apply(get_region)

    # Smart labeling: Only label the names if they are Top 200 in Times to prevent massive overlap
    plot_data['display_name'] = plot_data.apply(
        lambda x: x['name_clean_times'] if x['rank'] <= 200 else "", axis=1
    )

    # --- INTERACTIVE VISUALIZATION ---
    fig_size = px.scatter(
        plot_data, 
        x='student_population', 
        y='rank_advantage_qs',
        color='region_group',        # <--- Added Regional Colors!
        hover_name='name_clean_times', 
        text='display_name',         # <--- Added smart labels for screenshots
        hover_data=['country', 'rank', qs_rank_col],
        title='Size Bias Analysis: Student Population vs. QS Rank Advantage',
        labels={
            'student_population': 'Student Population (Log Scale)', 
            'rank_advantage_qs': 'Rank Advantage in QS (Positive = QS Favors, Negative = THE Favors)',
            'region_group': 'Geopolitical Region'
        },
        log_x=True, # Log Scale
        template='plotly_white',
        color_discrete_sequence=px.colors.qualitative.Prism, # Matched your color palette
        width=1200,  # Match the wide format of other charts
        height=900
    )
    
    # Add a zero line
    fig_size.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Neutral Rank (Both systems agree)")

    # Start with labels hidden, use font size 8
    fig_size.update_traces(mode='markers', textposition='top right', textfont=dict(size=8, color='#444'))

    # Add the Show/Hide Buttons at the BOTTOM LEFT
    fig_size.update_layout(
        margin=dict(b=120), 
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                buttons=list([
                    dict(label="Hide Names (Clean)",
                         method="update",
                         args=[{"mode": "markers"}]),  
                    dict(label="Show Top 200 Names",
                         method="update",
                         args=[{"mode": "markers+text"}]) 
                ]),
                showactive=True,
                x=0.0,
                xanchor="left",
                y=-0.15,      
                yanchor="top"
            ),
        ]
    )

    fig_size.show()

    print("\n--- GRAPH SUMMARY ---")
    large_unis = plot_data[plot_data['student_population'] > 20000]
    small_unis = plot_data[plot_data['student_population'] <= 20000]
    print(f"Average QS Rank Advantage for Large Universities (>20k): {large_unis['rank_advantage_qs'].mean():.2f}")
    print(f"Average QS Rank Advantage for Small Universities (<=20k): {small_unis['rank_advantage_qs'].mean():.2f}")

else:
    print(f"Could not find QS Rank column. Available columns: {df_merged.columns}")


--- GRAPH SUMMARY ---
Average QS Rank Advantage for Large Universities (>20k): 147.34
Average QS Rank Advantage for Small Universities (<=20k): 166.37


### Analysis of Insight 3: The "Size Bias" and Institutional Scaling Strategies

**Understanding the Graph:**

- **Y-Axis (Rank Advantage):** A positive value (above the red line) means the university performs **better in QS**. A negative value (below the red line) means it performs **better in Times Higher Education (THE)**.
- **X-Axis (Student Population):** Logarithmic scale representing the physical size of the institution.

**Macro-Observation: The "Boutique Premium" vs. The "Volume Engine"**
Contrary to the initial hypothesis, global ranking algorithms do **not** universally reward massive "Mega-Universities." In fact, the statistical summary reveals that smaller universities (≤20k students) hold a higher average QS advantage (+166 ranks) than large universities (+147 ranks).

However, by filtering the visual data by geopolitical region, we uncover three distinct, highly intentional national scaling strategies:

#### 1. Asia: The "Volume-to-Research" Pipeline (Teal Dots)

- **Visual Evidence:** In the Asian region, many massive institutions (30k–50k+ students) sink heavily *below* the red neutral line (indicating THE favors them significantly over QS).
- **Sociological Insight:** Asian policymakers (especially in China) frequently utilize university mergers to create massive state institutions. This scale naturally generates an enormous volume of raw research papers and patents, which the *Times Higher Education* algorithm heavily rewards. However, QS relies on global academic and employer surveys. The "Brand Reputation" of these massive new Asian universities has not yet permeated Western-dominated survey pools, resulting in a severe QS penalty.

#### 2. Europe: The "Historical Boutique Premium" (Blue Dots)

- **Visual Evidence:** The European cohort is tightly clustered in the mid-size range (10k–30k students), with a heavy upward pull *above* the red line (indicating a strong QS advantage).
- **Sociological Insight:** European institutions benefit from centuries of accumulated cultural prestige. Because QS weighs "Academic Reputation" at a massive 30%, these historic European universities enjoy a "Boutique Premium." They do not need to scale to 60,000 students to generate raw research volume; their historic brand name alone carries them to the top of the QS tables.

#### 3. The Anglosphere: The Public vs. Private Divide (Purple Dots)

- **Visual Evidence:** The Anglosphere is completely fractured. Massive state/public universities (e.g., *University of Massachusetts*, *Ohio State*) plummet below the line (THE favored). Meanwhile, smaller, elite private colleges (e.g., *Dartmouth*, *Brown*) float above the line (QS favored).
- **Sociological Insight:** This visualizes the duality of US/UK funding. Public universities use massive student populations to fund giant research labs (winning in THE metrics). Private Ivy League and elite UK institutions artificially restrict their student populations to maintain exclusivity, driving up their elite global reputation (winning in QS metrics).

### Actionable Policy Recommendation: Aligning Scale with Ranking Goals

For University Strategic Planners, this chart proves that **growth must be intentional**.

- **If the policy goal is to dominate the Times Higher Education (THE) rankings:** Policymakers should pursue institutional mergers and aggressively scale university populations to increase raw research volume, citations, and industry patents.
- **If the policy goal is to dominate the QS Rankings:** Planners should restrict student enrollment to maintain elite student-to-staff ratios, protect their historical exclusivity, and invest heavily in global marketing and international branding campaigns to artificially boost survey-based reputation scores.

------

In [11]:
# --- Insight 3 & Augmentation: Economic Power vs. Academic Standing (with API) ---
import requests
import pandas as pd
import plotly.express as px

# 1. Fetch Data
url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json&per_page=300&date=2023"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()[1]
    gdp_api_map = {entry['country']['value']: entry['value'] for entry in data if entry['value'] is not None}
else:
    gdp_api_map = {}

# 2. Data Cleaning: Name Normalization
name_fixer = {
    'United States': 'United States', 'United Kingdom': 'United Kingdom', 'Korea, Rep.': 'South Korea',
    'Hong Kong SAR, China': 'Hong Kong', 'Russian Federation': 'Russia', 'Egypt, Arab Rep.': 'Egypt',
    'Iran, Islamic Rep.': 'Iran', 'Turkiye': 'Turkey', 'Singapore': 'Singapore', 'China': 'China',
    'Japan': 'Japan', 'Germany': 'Germany', 'Switzerland': 'Switzerland', 'Australia': 'Australia',
    'Canada': 'Canada', 'Netherlands': 'Netherlands', 'Sweden': 'Sweden'
}
for wb_name, times_name in name_fixer.items():
    if wb_name in gdp_api_map:
        gdp_api_map[times_name] = gdp_api_map[wb_name]

df_merged['country_gdp'] = df_merged['country'].map(gdp_api_map)
country_performance = df_merged.groupby('country')[['rank', 'country_gdp']].median().dropna().reset_index()

# --- NEW: APPLY REGIONS FOR COLORING ---
def get_granular_region(country):
    if country in ['United States', 'Canada']: return 'North America'
    elif country in ['United Kingdom', 'Ireland']: return 'UK & Ireland'
    elif country in ['Australia', 'New Zealand']: return 'Oceania'
    elif country in ['China', 'Hong Kong', 'Macao', 'Taiwan']: return 'Greater China'
    elif country in ['Japan', 'South Korea']: return 'East Asia (Japan/Korea)'
    elif country in ['India', 'Pakistan', 'Bangladesh', 'Sri Lanka']: return 'South Asia'
    elif country in ['Singapore', 'Malaysia', 'Indonesia', 'Thailand', 'Vietnam', 'Philippines']: return 'Southeast Asia'
    elif country in ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain', 'Denmark', 'Finland', 'Norway', 'Austria', 'Luxembourg']: return 'Western/Northern Europe'
    elif country in ['Saudi Arabia', 'United Arab Emirates', 'Qatar', 'Israel', 'Iran', 'Turkey', 'Egypt', 'Lebanon', 'Jordan', 'Oman']: return 'Middle East & North Africa (MENA)'
    elif country in ['Brazil', 'Mexico', 'Chile', 'Argentina', 'Colombia', 'Peru']: return 'Latin America'
    elif country in ['South Africa', 'Ghana', 'Nigeria', 'Kenya']: return 'Sub-Saharan Africa'
    else: return 'Rest of Europe / Other'

country_performance['granular_region'] = country_performance['country'].apply(get_granular_region)


# --- PREPARE SPECIFIC LABELS ---
countries_to_label = [
    'United States', 'United Kingdom', 'China', 'Singapore', 'Switzerland', 
    'Germany', 'Australia', 'Canada', 'Japan', 'South Korea', 'Hong Kong', 
    'France', 'Netherlands', 'Sweden', 'India', 'Brazil', 'South Africa',
    'Luxembourg' # <--- Added Luxembourg!
]

# Create a new column: If country is in our list, use its name. Otherwise, leave it blank ("")
country_performance['display_label'] = country_performance['country'].apply(
    lambda x: x if x in countries_to_label else ""
)

# --- INTERACTIVE VISUALIZATION ---
fig = px.scatter(
    country_performance, 
    x='country_gdp', 
    y='rank', 
    color='granular_region', # <--- THIS COLORS THE DOTS AND MAKES THE INTERACTIVE LEGEND
    hover_name='country', 
    size='country_gdp',   
    text='display_label', 
    title='Economic Power (Live World Bank API Data) vs. Academic Standing',
    labels={
        'country_gdp': 'GDP Per Capita (USD - 2023)', 
        'rank': 'Median Times Rank (Lower is Better)',
        'granular_region': 'Geopolitical Region'
    },
    hover_data={'display_label': False, 'country_gdp': ':.0f'}, # <--- Hides the ugly "display_label=" in the hover box!
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Prism,
    width=1200, 
    height=900
)

# Move the text slightly to the right of the bubble
fig.update_traces(textposition='middle right', textfont=dict(size=11, color='black', weight='bold'))

# Add 15% padding to the right of the X-axis so Luxembourg's text doesn't get cut off
max_gdp = country_performance['country_gdp'].max()
fig.update_xaxes(range=[-5000, max_gdp * 1.15])

fig.update_yaxes(autorange="reversed") # Rank 1 at top
fig.show()

print("\n--- GRAPH SUMMARY ---")
gdp_rank_corr = country_performance['country_gdp'].corr(country_performance['rank'])
print(f"Correlation between GDP per Capita and Median Rank: {gdp_rank_corr:.2f}")


--- GRAPH SUMMARY ---
Correlation between GDP per Capita and Median Rank: -0.70


### Analysis of Augmentation: Economic Power vs. Academic Standing

**Data Source:** Live data fetched via **World Bank Open Data API** (Indicator: *GDP per capita, current US$*, 2023).

**Observation 1:**
The scatter plot demonstrates a **strong negative correlation (-0.70)** between a nation's wealth (GDP per capita) and its median university ranking.
* *Note: A negative correlation here is a positive outcome, as higher wealth leads to a lower (better) rank number.*
* This confirms that academic excellence is capital-intensive. Countries like **Switzerland** and **Singapore** (top right of the graph) exemplify this, translating massive per-capita wealth directly into elite academic standing.

**Observation 2:**
**China** appears as a significant outlier above the trend line.
* Despite having a GDP per capita (~$13k) comparable to developing nations, its median university ranking rivals that of developed European economies.
* This suggests that strategic state planning (e.g., "Double First Class" initiative) can override pure economic determinism, allowing a nation to build world-class universities before becoming fully "wealthy."

**Observation 3:**
Interestingly, the absolute wealthiest nations per capita (**Luxembourg** and **Ireland**) do not hold the top academic spots.
* Their median ranks are relatively modest (~265 and ~361, respectively).
* This indicates that while money is necessary for academic prestige, it is not sufficient. A deep national research ecosystem (like that of the **US**, **UK**, or **Germany**) requires population scale and historical infrastructure that small, finance-hub nations may lack.

------

In [12]:
# --- DEEP DIVE: Domain Analysis (Arts vs. Engineering) ---

import pandas as pd
import plotly.express as px

# 1. Load the European-formatted CSV
df_subs = pd.read_csv('../data/the_subjects_2026.csv', sep=';', decimal=',')

# 2. THE MATHEMATICAL ALGORITHM (Lexicographical Tie-Breaking)

# A. Get the official Bucket Rank (e.g., "401-500" becomes 401)
df_subs['rank_bucket'] = df_subs['rank'].astype(str).str.replace('=', '').str.split('–').str[0]
df_subs['rank_bucket'] = pd.to_numeric(df_subs['rank_bucket'], errors='coerce')

# B. Prepare the exact decimal columns
score_columns = [
    'scores_teaching', 'scores_research', 'scores_citations', 
    'scores_international_outlook', 'scores_industry_income'
]
for col in score_columns:
    df_subs[col] = df_subs[col].astype(str).str.replace(',', '.')
    df_subs[col] = pd.to_numeric(df_subs[col], errors='coerce')

# C. Calculate a simple sum purely to act as a Tie-Breaker
df_subs['tie_breaker_score'] = df_subs[score_columns].sum(axis=1)

# D. The Sort: Group by Subject, Sort by Official Rank FIRST, then Tie-Breaker SECOND
df_subs = df_subs.sort_values(
    by=['subject', 'rank_bucket', 'tie_breaker_score'], 
    ascending=[True, True, False] 
)

# E. Generate the final continuous rank (1, 2, 3... N) based on this precise sort
df_subs['continuous_rank'] = df_subs.groupby('subject').cumcount() + 1

# 3. Filter for Arts and Engineering
df_arts_eng = df_subs[df_subs['subject'].isin(['arts-and-humanities', 'engineering'])]

# 4. Pivot the data using our NEW CONTINUOUS RANK
df_pivot = df_arts_eng.pivot_table(
    index=['name', 'location'], 
    columns='subject', 
    values='continuous_rank'
).reset_index()

# Drop universities that don't teach both subjects
df_pivot = df_pivot.dropna(subset=['arts-and-humanities', 'engineering'])

# Add Region Colors (Make sure 'get_region' is defined above)
df_pivot['region_group'] = df_pivot['location'].apply(get_region)

# --- 5. INTERACTIVE VISUALIZATION ---
fig_domain = px.scatter(
    df_pivot,
    x='arts-and-humanities',
    y='engineering',
    color='region_group',
    hover_name='name',
    text='name',   # <--- Added names for screenshots
    hover_data=['location'],
    title='Domain Strategy: Engineering vs. Arts & Humanities (Top 1000)',
    labels={
        'arts-and-humanities': 'Arts Rank (Continuous)', 
        'engineering': 'Engineering Rank (Continuous)'
    },
    template='plotly_white',
    width=1200,  # <--- Made it wide!
    height=900   # <--- Made it taller for breathing room!
)

# Invert AND limit both axes to exactly 1000 to fill the screen
fig_domain.update_xaxes(range=[1000, 0])
fig_domain.update_yaxes(range=[1000, 0])

# Add a diagonal line exactly from 0 to 1000
fig_domain.add_shape(type="line", x0=0, y0=0, x1=1000, y1=1000, line=dict(color="red", dash="dash"))

# Start with labels hidden, use a smaller font size (8)
fig_domain.update_traces(mode='markers', textposition='top right', textfont=dict(size=8, color='#444'))

# Add the Show/Hide Buttons safely at the bottom left
fig_domain.update_layout(
    margin=dict(b=120), 
    updatemenus=[
        dict(
            type="buttons",
            direction="right", 
            buttons=list([
                dict(label="Hide Names (Clean)",
                     method="update",
                     args=[{"mode": "markers"}]),
                dict(label="Show All Names",
                     method="update",
                     args=[{"mode": "markers+text"}])
            ]),
            showactive=True,
            x=0.0,             
            xanchor="left",
            y=-0.15,           
            yanchor="top"
        ),
    ]
)

fig_domain.show()

## Geopolitical Strategies, Domain Bias, and Sociological Impacts

By utilizing interactive legend-filtering to deconstruct the "Overall Rank" into domain-specific performance (Engineering vs. Arts & Humanities), we uncover a profound shift in global higher education. The traditional ideal of the "Comprehensive University"—an institution equally elite across all disciplines—is now a luxury reserved almost exclusively for historic Anglosphere wealth. For the rest of the world, academia has fractured into highly intentional, asymmetric funding strategies.

### A Computational Sociology & Data Science Perspective
Before addressing regional policies, we must understand the socio-technical forces driving these visual patterns. From a data science and computational sociology perspective, global rankings suffer from severe methodological phenomena:

*   **The Flaw of Dimensionality Reduction:** Global rankings compress highly multidimensional data (teaching, patents, humanities research, engineering infrastructure) into a single 1D "Overall Score." As our scatter plot visually proves, this dimensionality reduction is mathematically misleading. It punishes highly specialized "Hidden Gems" (e.g., world-class technical institutes) by averaging their elite STEM scores with non-existent Arts scores, rendering them invisible in the Top 100.
*   **The Matthew Effect (Accumulated Advantage):** The Anglosphere's dominance in the top-right quadrant perfectly visualizes the sociological "Matthew Effect" (the rich get richer). Because institutions like Harvard and Oxford have historical wealth, they score highest in QS "Reputation." This reputation attracts more funding and top talent, which continuously inflates their scores in a self-fulfilling algorithmic feedback loop.
*   **Goodhart’s Law and Algorithmic Gaming:** *"When a measure becomes a target, it ceases to be a good measure."* The surge of Asian institutions above the red diagonal line proves these nations have reverse-engineered the *Times Higher Education* algorithm. Because THE heavily weights citations and industry patents (metrics naturally dominated by STEM), emerging nations are actively gaming the metric by aggressively defunding the humanities to optimize their engineering output.

### Regional Playbooks: Visual Evidence & Policy Recommendations
There is no "one-size-fits-all" approach to global rankings. Below are the strategic postures and actionable recommendations tailored for policymakers in four distinct geopolitical regions, based strictly on the visual evidence in our interactive dashboard.

#### 1. The Anglosphere: "The Endowment Monopoly"
*   **Visual Evidence:** When isolated, Anglosphere universities (US, UK, Australia) dominate the absolute top-right quadrant (Ranks 1–100) and tightly hug the balanced diagonal red line. 
*   **Strategic Posture:** These nations operate on the "Comprehensive Titan" model. Powered by multi-billion-dollar historical endowments and the global dominance of English-language publishing, they are the only region that can afford to maintain elite status in both capital-intensive (Engineering) and human-capital-intensive (Arts) domains simultaneously.
*   **💡 Actionable Policy Recommendation:** Anglosphere policymakers must shift from an offensive to a *defensive* strategy. While their Arts/Humanities prestige is currently untouchable, they are facing aggressive incursions from Asian technical institutes in STEM. Funding must be urgently ring-fenced for applied sciences to prevent a slow erosion of their monopoly at the very top of the rankings.

#### 2. Asia (Major Economies): "The Asymmetric Tech Vanguard"
*   **Visual Evidence:** The Asian cluster severely skews *above* the balanced diagonal line, heavily populating the upper-left quadrant (elite in Engineering, trailing in Arts).
*   **Strategic Posture:** Nations like China, Singapore, and South Korea are successfully executing an asymmetric funding strategy. They are intentionally starving generalist humanities programs to funnel massive state subsidies directly into STEM, AI, and Engineering to optimize ROI on algorithmic ranking metrics. 
*   **💡 Actionable Policy Recommendation:** Asian policymakers should continue this hyper-specialization to quickly dominate the Top 50 global ranks. However, to break the ultimate "Glass Ceiling" (entering the Top 10 globally), these nations must eventually pivot to building "Soft Power." Long-term policy must begin slowly injecting capital into the Arts & Humanities to build the holistic, historical prestige that currently keeps Harvard and Oxford at number one.

#### 3. The European Union: "The Dual-Track Heritage Model"
*   **Visual Evidence:** The EU shows a highly dispersed but bifurcated pattern: a cluster of dots heavily above the line (Specialized Polytechnics) and a cluster of dots below the line (Classic/Historical Universities).
*   **Strategic Posture:** Europe is caught between honoring its deep historical heritage and competing in the modern tech economy. Unlike the US, which merges both into single mega-universities, the EU naturally splits its talent into distinct, specialized institutions (e.g., TU Munich for tech, LMU Munich for humanities).
*   **💡 Actionable Policy Recommendation:** EU policymakers should stop forcing their institutions to chase Anglosphere "generalist" ranking metrics. Funding should be formally bifurcated: protect and fund "Heritage Universities" purely for their cultural and academic prestige (optimizing for QS metrics), while aggressively scaling funding for their "TU/Polytechnic Networks" to act as the economic and industrial engines of the continent (optimizing for THE metrics).

#### 4. Rest of World: "Overcoming the Capital Barrier"
*   **Visual Evidence:** Emerging economies are trapped in the bottom-left quadrant. Visually, they suffer a severe "sag" below the diagonal line, meaning they rank moderately in Arts (e.g., Top 500) but completely fall off the charts in Engineering (Rank 800+).
*   **Strategic Posture:** Developing nations are hitting the "Capital Barrier." World-class engineering requires supercomputers, biotech labs, and advanced robotics—infrastructure that emerging economies simply cannot afford to scale globally compared to wealthy nations.
*   **💡 Actionable Policy Recommendation:** Policymakers in emerging economies must abandon broad, general engineering programs that cannot compete with the Anglosphere. Instead, they must niche down aggressively. By securing foreign corporate partnerships and focusing exclusively on highly localized, specialized applied sciences (e.g., agricultural tech in South America, water management tech in Africa), they can bypass the general capital barrier and create isolated pockets of global excellence.

------

In [13]:
# --- ADVANCED DEEP DIVE: Granular Sub-Regional Strategy ---

# 1. Define a highly granular sub-regional mapping function
def get_granular_region(country):
    # Split Anglosphere
    if country in ['United States', 'Canada']: return 'North America'
    elif country in ['United Kingdom', 'Ireland']: return 'UK & Ireland'
    elif country in ['Australia', 'New Zealand']: return 'Oceania'
    # Split Asia
    elif country in ['China', 'Hong Kong', 'Macao', 'Taiwan']: return 'Greater China'
    elif country in ['Japan', 'South Korea']: return 'East Asia (Japan/Korea)'
    elif country in ['India', 'Pakistan', 'Bangladesh', 'Sri Lanka']: return 'South Asia'
    elif country in ['Singapore', 'Malaysia', 'Indonesia', 'Thailand', 'Vietnam', 'Philippines']: return 'Southeast Asia'
    # Split Rest of World & Europe
    elif country in ['Germany', 'France', 'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 'Italy', 'Spain', 'Denmark', 'Finland', 'Norway', 'Austria']: return 'Western/Northern Europe'
    elif country in ['Saudi Arabia', 'United Arab Emirates', 'Qatar', 'Israel', 'Iran', 'Turkey', 'Egypt', 'Lebanon', 'Jordan', 'Oman']: return 'Middle East & North Africa (MENA)'
    elif country in ['Brazil', 'Mexico', 'Chile', 'Argentina', 'Colombia', 'Peru']: return 'Latin America'
    elif country in ['South Africa', 'Ghana', 'Nigeria', 'Kenya']: return 'Sub-Saharan Africa'
    else: return 'Rest of Europe / Other'

# Apply the new mapping
df_pivot['granular_region'] = df_pivot['location'].apply(get_granular_region)

# 2. INTERACTIVE VISUALIZATION
import plotly.express as px

fig_granular = px.scatter(
    df_pivot,
    x='arts-and-humanities',
    y='engineering',
    color='granular_region',
    hover_name='name',
    text='name',            # <--- Reverted to showing ALL university names!
    hover_data=['location'],
    title='Micro-Geopolitics: Granular Domain Strategy (Top 1000)',
    labels={
        'arts-and-humanities': 'Arts Rank (Continuous)', 
        'engineering': 'Engineering Rank (Continuous)'
    },
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Prism, 
    width=1200,             # <--- Made the chart much WIDER horizontally
    height=900              # <--- Made it taller to give data more breathing room
)

# Invert and limit axes
fig_granular.update_xaxes(range=[1000, 0])
fig_granular.update_yaxes(range=[1000, 0])

# Add the balanced diagonal line
fig_granular.add_shape(type="line", x0=0, y0=0, x1=1000, y1=1000, line=dict(color="red", dash="dash"))

# Start with labels hidden, use a smaller font size (8) so they fit better when crowded
fig_granular.update_traces(mode='markers', textposition='top right', textfont=dict(size=8, color='#444'))

# Add the Show/Hide Buttons (Kept at the BOTTOM LEFT so they don't cover the title)
fig_granular.update_layout(
    margin=dict(b=120), # Add extra space at the bottom for the buttons
    updatemenus=[
        dict(
            type="buttons",
            direction="right", 
            buttons=list([
                dict(label="Hide Names (Clean)",
                     method="update",
                     args=[{"mode": "markers"}]),
                dict(label="Show All Names",
                     method="update",
                     args=[{"mode": "markers+text"}])
            ]),
            showactive=True,
            x=0.0,             
            xanchor="left",
            y=-0.15,           # Keeps the button safely below the X-axis
            yanchor="top"
        ),
    ]
)

fig_granular.show()

# Micro-Geopolitics and the Sociology of Global Academia

### A Visual Analysis of National Funding Strategies (Top 1000)

By applying Intra-Bin Interpolation to calculate a true continuous rank, our dashboard exposes the structural realities of global academia. Through isolating specific granular regions, we observe that the "Comprehensive University" model is not a global standard, but rather a regional luxury.

For Higher Education Policymakers, four distinct strategic paradigms emerge from the visual data:

### 1. Institutional Isomorphism: The Anglosphere & Oceania

- **Visual Evidence:** In the **UK & Ireland** and **Oceania** screenshots, the data points adhere almost perfectly to the red diagonal baseline. There are virtually zero extreme outliers. **North America** displays a similar trend but with massive volume, anchoring the absolute (0,0) coordinate.
- **Computational Sociology Insight:** This perfect linear distribution is a textbook example of **"Institutional Isomorphism"**—the sociological process where organizations in the same environment begin to copy each other and look identical. Because universities in the UK (via the REF framework) and Australia rely on highly standardized, centralized government funding models, they scale symmetrically.
- **Policy Takeaway:** Policymakers in these regions are funding *balance*. A wealthy Australian or UK university will be equally elite in Arts and Engineering; a mid-tier university will be equally mid-tier in both. They do not sacrifice one domain for the other because their domestic funding algorithms punish imbalance.

### 2. The State-Capital Tech Vanguard: Greater China & MENA

- **Visual Evidence:** The **Greater China** (Red) and **Middle East & North Africa** (Green) screenshots share a nearly identical, highly asymmetric visual profile. The vast majority of their universities float high *above* the red diagonal line, indicating Elite Engineering ranks paired with lagging Arts ranks.
- **Computational Sociology Insight:** This is the visual signature of **Algorithmic Gaming** fueled by state capital. Both regions possess massive sovereign wealth or state-directed GDP, which they are injecting directly into CapEx-heavy STEM fields. By buying advanced laboratories and recruiting elite engineering faculty, they are successfully "hacking" the Times Higher Education metrics (which heavily reward patents and citations) to leapfrog historic Western institutions.
- **Policy Takeaway:** For MENA and Chinese policymakers, Arts and Humanities are viewed as low-ROI investments for global ranking mobility. Their strategy proves that hyper-specialization in Engineering is the fastest ladder to global academic relevance for non-Western nations.

### 3. The "Capital Barrier" Trap: Latin America & Sub-Saharan Africa

- **Visual Evidence:** In both the **Latin America** (Teal) and **Sub-Saharan Africa** (Purple) screenshots, the data clusters almost entirely *below* the red diagonal line. Universities here rank moderately well in Arts & Humanities (Ranks 200–600) but drop precipitously in Engineering (Ranks 600–1000+).
- **Computational Sociology Insight:** This visualizes the harsh macroeconomic reality of the **Global South Capital Barrier**. Maintaining a top-tier Arts faculty requires Operational Expenditure (OpEx)—libraries, salaries, and human capital. Competing in global Engineering requires massive Capital Expenditure (CapEx)—robotics, supercomputers, and biotech facilities.
- **Policy Takeaway:** Policymakers in the Global South cannot currently afford to compete in the global STEM arms race. They are relying on historic, cultural prestige to maintain global visibility. To break this trap, African and Latin American ministries must pivot from general public funding to securing localized, multinational corporate partnerships to subsidize their STEM infrastructure.

### 4. The "Missing Data" Insight: South Asia and Extreme Specialization

- **Visual Evidence:** In the **South Asia** screenshot, the graph is almost entirely empty. Only a handful of comprehensive universities (like Jamia Millia Islamia and Jadavpur) appear, and they sit below the red line.
- **Data Science Insight:** Where are the world-famous Indian Institutes of Technology (IITs)? **They are missing from the graph because they were dropped by our Data Science cleaning pipeline.** By requiring a university to have *both* an Arts rank and an Engineering rank to be plotted on this comparative axis, the code naturally eliminated the IITs, which exclusively teach STEM.
- **Policy Takeaway:** This exposes a massive blind spot in how global rankings judge academia. Rankings naturally punish "Pure Specialists." Indian policymakers have created some of the best engineering talent pipelines on earth, but because those institutes do not teach Humanities, they cannot compete on comprehensive global ranking tables. Policymakers must recognize that global rankings are heavily biased toward the Western "Comprehensive" university model.

------

In [14]:
# --- EU CASE STUDY: The European Higher Education Landscape ---

import plotly.express as px

# 1. Filter the dataset for strictly European countries (Assumes df_pivot is loaded)
european_countries = [
    'United Kingdom', 'Germany', 'France', 'Italy', 'Spain', 
    'Netherlands', 'Switzerland', 'Sweden', 'Belgium', 
    'Denmark', 'Finland', 'Norway', 'Austria', 'Ireland', 
    'Portugal', 'Poland', 'Greece', 'Czech Republic'
]

df_europe = df_pivot[df_pivot['location'].isin(european_countries)].copy()

# 2. Create a specific European mapping to highlight national systems
def get_eu_system(country):
    if country == 'United Kingdom': return 'United Kingdom'
    elif country == 'Germany': return 'Germany (TU & Excellence Model)'
    elif country == 'France': return 'France (Grandes Écoles & ComUEs)'
    elif country in ['Italy', 'Spain', 'Portugal', 'Greece']: return 'Southern Europe (Historic Comprehensives)'
    elif country in ['Netherlands', 'Belgium', 'Switzerland', 'Austria']: return 'Western/Alpine Hubs'
    elif country in ['Sweden', 'Denmark', 'Finland', 'Norway']: return 'Nordics'
    else: return 'Eastern/Central Europe'

df_europe['eu_system'] = df_europe['location'].apply(get_eu_system)

# 3. INTERACTIVE VISUALIZATION
fig_eu = px.scatter(
    df_europe,
    x='arts-and-humanities',
    y='engineering',
    color='eu_system',
    hover_name='name',
    text='name',   
    hover_data=['location'],
    title='European Case Study: National Strategies in Engineering vs. Arts',
    labels={
        'arts-and-humanities': 'Arts Rank (Continuous)', 
        'engineering': 'Engineering Rank (Continuous)'
    },
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold,
    width=1200,   # <--- Made it wide!
    height=900    # <--- Made it taller!
)

# Invert and limit axes
fig_eu.update_xaxes(range=[800, 0])
fig_eu.update_yaxes(range=[800, 0])

# Add the balanced diagonal line
fig_eu.add_shape(type="line", x0=0, y0=0, x1=800, y1=800, line=dict(color="gray", dash="dash"))

# Start with labels hidden, use font size 8 for consistency
fig_eu.update_traces(mode='markers', textposition='top right', textfont=dict(size=8, color='#444'))

# Add the Show/Hide Buttons at the BOTTOM LEFT (Fixes title overlap)
fig_eu.update_layout(
    margin=dict(b=120), 
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=list([
                dict(label="Hide Names (Clean)",
                     method="update",
                     args=[{"mode": "markers"}]),  
                dict(label="Show Names (For Screenshots)",
                     method="update",
                     args=[{"mode": "markers+text"}]) 
            ]),
            showactive=True,
            x=0.0,
            xanchor="left",
            y=-0.15,      # <--- Safely below the chart!
            yanchor="top"
        ),
    ]
)

fig_eu.show()

### The Fractured European Higher Education Area

While the European Union promotes a unified "European Higher Education Area" (EHEA) through initiatives like the Bologna Process, our interactive domain analysis reveals that **national funding strategies remain deeply fractured.**

By plotting Engineering against Arts & Humanities, we uncover three distinct geopolitical strategies currently operating within Europe:

#### 1. The "Algorithmic Optimizers" (France)

- **The Visual Evidence:** France possesses very few dots compared to the UK or Germany, but its top institutions (e.g., *Université Paris-Saclay, Sorbonne University*) sit at the absolute pinnacle of the top-right quadrant.
- **The Policy Context:** Ten years ago, France performed terribly in global rankings. Why? Because the historic French system was highly fragmented, splitting elite specialized schools (*Grandes Écoles*) away from massive public universities.
- **The Actionable Insight:** The French government recognized that THE and QS algorithms heavily reward massive, comprehensive "mega-universities" (like Harvard). In response, they launched the *Initiatives d'Excellence (IdEx)*, forcing dozens of smaller institutes to merge into massive consortiums (ComUEs). The dots in the top right of our graph are the direct result of this policy: France artificially engineered "Generalist Titans" by combining their specialized tech schools and historic humanities colleges under single, unified ranking profiles.

#### 2. The "Industrial Engines" (Germany & The Nordics)

- **The Visual Evidence:** Institutions in Germany (e.g., *TU Munich, KIT*) and the Nordics (e.g., *KTH Royal Institute of Technology, DTU*) sit exceptionally high above the red diagonal line, indicating elite Engineering ranks (Top 50) paired with lagging Arts ranks (300+).
- **The Policy Context:** This is not a failure of their Arts departments; it is a deliberate, state-sponsored macroeconomic strategy. Germany’s *Exzellenzstrategie* (Excellence Strategy) and Nordic innovation funds are intentionally funneled into Technical Universities ("TUs") to support their national export economies (automotive, green tech, and advanced manufacturing).
- **The Actionable Insight:** Policymakers in mid-sized, export-driven economies should study the German/Nordic model. Rather than spreading limited GDP across all subjects to chase Anglosphere prestige, they achieve a higher national Return on Investment (ROI) by hyper-funding specialized STEM infrastructure.

#### 3. The "Regulated Generalists" (United Kingdom)

- **The Visual Evidence:** The UK dots form a remarkably tight, perfectly 45-degree line directly over the red dashed diagonal. There are almost zero outliers.
- **The Policy Context:** The UK higher education system is steered by a centralized government funding mechanism called the **Research Excellence Framework (REF)**. The REF evaluates and funds universities based on comprehensive performance.
- **The Actionable Insight:** Because UK universities rely on this unified government funding, they scale proportionally. A wealthy UK university (like Cambridge) can afford to be elite at everything. A mid-tier UK university will be exactly mid-tier in both Arts and Engineering. The visual tightness of the UK dots proves that centralized, standardized funding models prevent universities from adopting the asymmetric "tech-only" strategies seen in Asia or Germany.

#### 4. The "Capital Trap" vs. "Wealthy Hubs" (Southern Europe vs. Alpine/Benelux)

- **The Visual Evidence:** There is a stark north-south divide. Wealthy Alpine/Western hubs (Switzerland, Netherlands, Belgium—including **KU Leuven**) dominate the top-right box. Conversely, Southern Europe (Italy, Spain, Greece) sags heavily into the bottom-right quadrant (Humanities dominant, Engineering lagging).
- **The Policy Context:** Modern Engineering research requires massive continuous Capital Expenditure (CapEx) for supercomputers, robotics, and biotech labs. Arts and Humanities rely on Operational Expenditure (OpEx) and historic libraries.
- **The Actionable Insight:** Following the 2008 Eurozone crisis, Southern Europe faced severe austerity, capping their ability to buy elite STEM infrastructure. They are currently surviving in the global rankings solely by leaning on their centuries-old cultural prestige (e.g., *University of Bologna*). Conversely, high-GDP nations like Switzerland and Belgium have the sheer economic power to fund world-class engineering labs *without* sacrificing their historic humanities prestige.

---------

## General Conclusions & Actionable Policy Insights

In this project, we analyzed the global higher education landscape to identify key strategic trends, applying advanced data science techniques such as data cleaning, fuzzy merging, API integration, and intra-bin interpolation. For University Strategic Planners and Educational Policymakers, four distinct global realities emerge:

### 1. Longitudinal Trends (The "Eastward Shift")
Using 10 years of *Times Higher Education* data, we identified a structural shift in the global academic elite. The "Asian Tigers" (China, Singapore, Hong Kong) have more than doubled their presence in the Top 200 since 2016, eroding the historic dominance of the Anglosphere (US/UK).

### 2. Methodological Bias (Prestige vs. Performance)
By comparing the *QS 2026* and *Times 2026* datasets, the analysis uncovered significant methodological divergences:
* The analysis revealed that US and UK universities show high agreement between ranking systems. However, Asian and "Rest of World" institutions face a "Methodological Divergence," where their standing fluctuates significantly depending on whether the system prioritizes reputation (QS) or citations (Times).
* Using Z-Score standardization, the project identified that "Old Money" institutions (e.g., LSE, Harvard) rely heavily on Brand Reputation, whereas "Hidden Engines" in East Asia (e.g., Taiwan, Japan) produce world-class industrial output but suffer from a "Brand Deficit."
* Contrary to the initial hypothesis that "Bigger is Better," the data revealed that **smaller universities** actually held a slightly higher rank advantage in QS than larger ones. This suggests that QS's methodology may favor focused, specialized institutions just as much as large global brands, whereas the Times's metrics may penalize low-volume research output.

### 3. Socio-Economic Factors (API Augmentation)
By augmenting the dataset with live economic data from the **World Bank API**, we confirmed a general correlation between national wealth and academic performance. However, **China remains a massive outlier**, achieving elite rankings despite a significantly lower GDP per capita than its Western rivals. This suggests that strategic government policy, combined with a widespread societal emphasis on higher education and its prestige, can defy pure economic determinism.

### 4. Micro-Geopolitics & Domain Strategy (Arts vs. Engineering)
By applying lexicographical tie-breaking algorithms to unmask the true continuous distribution of THE Subject Rankings, we discovered that the "Comprehensive University" model is a regional luxury, not a global standard. Global academia has fractured into highly intentional, asymmetric funding strategies:
* **The Anglosphere "Halo Effect":** US and UK institutions rely on multi-billion dollar endowments to achieve elite, balanced performance across both Engineering and the Humanities, creating an insurmountable monopoly at the top.
* **The "Asymmetric Tech Vanguard":** Nations in Asia (China, Singapore) and Northern Europe (Germany, Nordics) are intentionally sacrificing generalist prestige in the Arts to aggressively funnel state subsidies into capital-intensive STEM infrastructure, maximizing their ROI in global ranking algorithms.
* **The "Capital Barrier":** Emerging economies in the Global South (Latin America, Sub-Saharan Africa), alongside Southern Europe, remain trapped below the threshold of global Engineering competitiveness due to the massive capital expenditure (CapEx) required for modern labs. Instead, they must rely on historic, human-capital-driven Humanities prestige to maintain global visibility.
* **Final Policy Recommendation:** To break into the Global Top 100, policymakers outside of the US/UK must abandon the "Generalist Titan" model. The data proves that hyper-specialization—either through highly-funded national tech hubs or aggressive corporate STEM partnerships—is the only viable strategy to overcome the Anglosphere's historic wealth advantage.

---
**Methodological Disclaimer:** 
These metrics should be interpreted with caution. As noted in a recent report by the *South China Morning Post* (2025, cf. [source](https://www.scmp.com/news/china/science/article/3332679/qs-university-rankings-ridiculed-china-latest-listing-bombs-online)), ranking outcomes can be volatile and disconnected from perceived academic reality. The report highlights that the *QS 2026* data faced significant backlash for "unintentionally funny" anomalies, such as ranking lesser-known Malaysian universities above research giants like the University of Tokyo. This suggests that in some cases, high rankings may reflect how well institutions tailor their strategies to specific metrics (such as "Internationalization" or reputational surveys) rather than indicating genuine academic superiority, supporting the view that commercial rankings can be "inevitably inflated."